# Módulo EP3 — Observabilidad Sanitizada, Benchmark Offline y Dashboard

Correspondencia directa con:
- `app/observability/telemetry.py` y `app/observability/store.py`
- `app/evaluation/agentops_runner.py` y `app/evaluation/agentops_metrics.py`
- `app/api/dashboard_routes.py` y `app/api/observability_routes.py`


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HikariLucy/KnowledgeFlow-AgentOps.git"
REPO_BRANCH = "feat/eft-notebook"

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "app" / "main.py").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None and IN_COLAB:
    target = Path("/content/KnowledgeFlow-AgentOps")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(target)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(target / "requirements.txt")], check=True)
    repo_root = target

if repo_root is None:
    raise RuntimeError("No se encontró la raíz de KnowledgeFlow-AgentOps. Ejecuta desde el repositorio o en Google Colab.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Seguridad: modo offline sin claves de Gemini por defecto
os.environ.setdefault("OBS_EXECUTION_MODE", "offline")
os.environ.pop("GEMINI_API_KEY", None)
RUN_LIVE_GEMINI = False

print(f"Entorno listo | IN_COLAB={IN_COLAB} | repo_root={repo_root} | RUN_LIVE_GEMINI={RUN_LIVE_GEMINI}")


## 1. Inspección fiel del código fuente EP3 (sin duplicación manual)


In [ ]:
import inspect
from app.observability import telemetry
from app.api import dashboard_routes

print("=== app/observability/telemetry.py :: error_code & attributes ===")
print(inspect.getsource(telemetry.error_code))
print(inspect.getsource(telemetry.attributes))
print("=== app/api/dashboard_routes.py :: local_access ===")
print(inspect.getsource(dashboard_routes.local_access))


## 2. Trazas sanitizadas, benchmark offline y prueba del dashboard en SQLite temporal


In [ ]:
import base64
from tempfile import TemporaryDirectory
from fastapi.testclient import TestClient
from pydantic import SecretStr
from app.core.config import Settings
from app.observability import telemetry
from app.observability.store import EventStore
from app.evaluation.agentops_dataset import load_agentops_dataset
from app.evaluation.agentops_runner import run_agentops_benchmark
import app.main as app_main

prev_obs = telemetry.observer
prev_set = getattr(app_main.app.state, "observability_settings", None)
prev_sto = getattr(app_main.app.state, "observability_store", None)

with TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    store = EventStore(tmp_path / "obs.sqlite3")
    qdir = tmp_path / "quality"
    skey = b"ep3-notebook-session-key-32bytes"
    rkey = "ep3-read-key-32-bytes-minimum-len"
    dpass = "ep3-dashboard-pass-32-chars-min!"

    telemetry.observer = telemetry.Observer(enabled=True, mode="offline", session_key=skey, store=store)
    with telemetry.span("agent", "execute") as sp:
        with telemetry.span("tool", "search_knowledge"):
            pass

    dataset = load_agentops_dataset(repo_root / "evaluation" / "agentops" / "dataset_v1.json")
    report = run_agentops_benchmark(dataset, mode="offline", repeats=2, output_dir=qdir)

    from datetime import datetime, timedelta, timezone
    now_utc = datetime.now(timezone.utc)
    since_iso = (now_utc - timedelta(hours=1)).isoformat().replace("+00:00", "Z")
    until_iso = (now_utc + timedelta(minutes=5)).isoformat().replace("+00:00", "Z")

    hdr = {"Authorization": "Basic " + base64.b64encode(f"observer:{dpass}".encode()).decode()}
    with TestClient(app_main.app, base_url="http://127.0.0.1:8000", client=("127.0.0.1", 50000)) as client:
        app_main.app.state.observability_settings = Settings(
            obs_enabled=True,
            obs_execution_mode="offline",
            obs_session_key=SecretStr(skey.decode()),
            obs_store_enabled=True,
            obs_store_path=str(tmp_path / "obs.sqlite3"),
            obs_read_key=SecretStr(rkey),
            obs_dashboard_enabled=True,
            obs_dashboard_password=SecretStr(dpass),
            obs_quality_dir=str(qdir),
        )
        app_main.app.state.observability_store = store
        unauth = client.get("/dashboard")
        auth_ok = client.get(f"/dashboard/api/summary?execution_mode=offline&since={since_iso}&until={until_iso}", headers=hdr)
        qual_ok = client.get("/dashboard/api/quality?execution_mode=offline", headers=hdr)

    print("Benchmark offline:", report["metrics"]["task_accuracy"]["value"], "synthetic:", report["synthetic"])
    print("Dashboard sin auth ->", unauth.status_code, "| con Basic auth ->", auth_ok.status_code)
    print("Quality endpoint ->", qual_ok.status_code, "| sample_size:", qual_ok.json()["benchmarks"][0]["sample_size"])

telemetry.observer = prev_obs
app_main.app.state.observability_settings = prev_set
app_main.app.state.observability_store = prev_sto
